In [2]:
!pip install bleak

Defaulting to user installation because normal site-packages is not writeable
  Using cached bleak-2.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached winrt_windows_devices_bluetooth-3.2.1-cp312-cp312-win_amd64.whl.metadata (1.6 kB)
  Using cached winrt_windows_devices_bluetooth_advertisement-3.2.1-cp312-cp312-win_amd64.whl.metadata (1.4 kB)
  Using cached winrt_windows_devices_bluetooth_genericattributeprofile-3.2.1-cp312-cp312-win_amd64.whl.metadata (1.5 kB)
  Using cached winrt_windows_devices_enumeration-3.2.1-cp312-cp312-win_amd64.whl.metadata (1.5 kB)
  Using cached winrt_windows_devices_radios-3.2.1-cp312-cp312-win_amd64.whl.metadata (1.1 kB)
  Using cached winrt_windows_foundation-3.2.1-cp312-cp312-win_amd64.whl.metadata (1.0 kB)
  Using cached winrt_windows_foundation_collections-3.2.1-cp312-cp312-win_amd64.whl.metadata (1.1 kB)
  Using cached winrt_windows_storage_streams-3.2.1-cp312-cp312-win_amd64.whl.metadata (1.3 kB)
  Using cached winrt_runtime-3.2.1-cp312-cp312-win_


[notice] A new release of pip is available: 25.0.1 -> 26.0
[notice] To update, run: C:\Users\youss\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import struct
import pandas as pd
import os

# ==========================================
# USER CONFIGURATION
# You MUST get these numbers from the Arduino Serial Monitor output
# Example output: "PPG samples collected: 8150"
# ==========================================
PPG_COUNT = 2270  # <--- REPLACE THIS with your actual PPG count
IMU_COUNT = 500  # <--- REPLACE THIS with your actual IMU count

INPUT_FILE = "captured_data.bin"
OUTPUT_PPG = "clean_ppg_data.csv"
OUTPUT_IMU = "clean_imu_data.csv"

def parse_bin_file():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found.")
        return

    print(f"Processing {INPUT_FILE}...")
    
    with open(INPUT_FILE, "rb") as f:
        data = f.read()

    offset = 0
    total_size = len(data)
    
    # --- 1. Parse PPG Sensor 1 ---
    # Format: Big Endian (>), Unsigned Int (I)
    ppg1_data = []
    print("Extracting PPG Sensor 1...")
    for _ in range(PPG_COUNT):
        if offset + 4 > total_size: break
        val = struct.unpack_from('>I', data, offset)[0]
        ppg1_data.append(val)
        offset += 4

    # --- 2. Parse PPG Sensor 2 ---
    # Format: Big Endian (>), Unsigned Int (I)
    ppg2_data = []
    print("Extracting PPG Sensor 2...")
    for _ in range(PPG_COUNT):
        if offset + 4 > total_size: break
        val = struct.unpack_from('>I', data, offset)[0]
        ppg2_data.append(val)
        offset += 4

    # --- 3. Parse IMU Data ---
    # Format: Little Endian (<), 6 Floats (ffffff)
    imu_records = []
    print("Extracting IMU Data...")
    for _ in range(IMU_COUNT):
        if offset + 24 > total_size: break
        # Returns tuple: (ax, ay, az, gx, gy, gz)
        vals = struct.unpack_from('<ffffff', data, offset)
        imu_records.append(vals)
        offset += 24

    # --- 4. Parse Temperature ---
    # Format: Little Endian (<), Float (f)
    temperature = None
    if offset + 4 <= total_size:
        temperature = struct.unpack_from('<f', data, offset)[0]
        print(f"Extracting Temperature... {temperature:.2f} °C")
    else:
        print("Warning: unexpected end of file before temperature data.")

    # ==========================================
    # SAVE TO CSV
    # ==========================================
    
    # Save PPG Data (High Frequency)
    df_ppg = pd.DataFrame({
        'Sample_Num': range(len(ppg1_data)),
        'PPG_Sensor1_Raw': ppg1_data,
        'PPG_Sensor2_Raw': ppg2_data
    })
    df_ppg.to_csv(OUTPUT_PPG, index=False)
    print(f"Saved PPG data to {OUTPUT_PPG} ({len(df_ppg)} rows)")

    # Save IMU Data (Lower Frequency)
    # We separate this because IMU has 50Hz rate vs PPG ~400Hz
    df_imu = pd.DataFrame(imu_records, columns=['AccelX', 'AccelY', 'AccelZ', 'GyroX', 'GyroY', 'GyroZ'])
    
    # Add temperature to the IMU file (just repeats the value or adds as metadata)
    df_imu['Avg_Temperature_C'] = temperature if temperature is not None else 0.0
    
    df_imu.to_csv(OUTPUT_IMU, index=False)
    print(f"Saved IMU data to {OUTPUT_IMU} ({len(df_imu)} rows)")

if __name__ == "__main__":
    try:
        parse_bin_file()
    except Exception as e:
        print(f"An error occurred: {e}")
        print("Tip: Double check your PPG_COUNT and IMU_COUNT match the Serial Monitor exactly.")

Processing captured_data.bin...
Extracting PPG Sensor 1...
Extracting PPG Sensor 2...
Extracting IMU Data...
Extracting Temperature... 1743028944488917440434070880256.00 °C
Saved PPG data to clean_ppg_data.csv (2270 rows)
Saved IMU data to clean_imu_data.csv (14 rows)
